# Deliverable C — Data Visualizations

**Goal:** Create publication-quality charts that communicate key patterns in ride demand.

**Outputs:**
- C1: Time series plots (daily/weekly aggregates)
- C2: Hourly patterns (by day of week)
- C3: Zone comparison charts
- C4: Weather impact visualizations
- C5: Event impact visualizations
- C6: Correlation heatmaps
- C7: Distribution plots

**Data source:** `data/processed/master_train.csv`

**Figure outputs:** Saved to `figures/` directory

## 0 · Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Paths
ROOT = Path('..').resolve()
PROC = ROOT / 'data' / 'processed'
FIG = ROOT / 'figures'

FIG.mkdir(parents=True, exist_ok=True)

# Plotting configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10

# Color palette
COLORS = {
    'primary': '#2E86AB',
    'secondary': '#A23B72',
    'accent': '#F18F01',
    'success': '#06A77D',
    'warning': '#D9514E',
    'neutral': '#5E6472'
}

print(f'ROOT: {ROOT}')
print(f'FIGURES: {FIG}')
print('\nPlotting environment ready ✓')

## 1 · Load Data

In [ ]:
# Load master train data
df = pd.read_csv(PROC / 'master_train.csv', parse_dates=['ts'])

print(f'Data loaded: {df.shape}')
print(f'Date range: {df["ts"].min()} to {df["ts"].max()}')
print(f'Zones: {sorted(df["zone"].unique())}')

# Add helper columns
df['date'] = df['ts'].dt.date
df['hour'] = df['ts'].dt.hour
df['dayofweek'] = df['ts'].dt.dayofweek
df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)

print('\n✓ Data prepared for visualization')

---
## C1 · Time Series Plots

In [ ]:
print('=== C1: TIME SERIES VISUALIZATIONS ===')

# Daily aggregates (citywide)
daily = df.groupby('date')['trips'].sum().reset_index()
daily['date'] = pd.to_datetime(daily['date'])
daily['ma7'] = daily['trips'].rolling(window=7, center=True).mean()

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(daily['date'], daily['trips'], alpha=0.3, color=COLORS['neutral'], label='Daily total')
ax.plot(daily['date'], daily['ma7'], linewidth=2, color=COLORS['primary'], label='7-day moving average')

ax.set_title('Citywide Daily Ride Demand (All Zones Combined)', fontsize=16, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Total Trips', fontsize=12)
ax.legend(loc='best')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'c01_daily_demand_timeseries.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c01_daily_demand_timeseries.png')

In [ ]:
# Weekly patterns (by zone)
weekly_by_zone = df.groupby(['date', 'zone'])['trips'].sum().reset_index()
weekly_by_zone['date'] = pd.to_datetime(weekly_by_zone['date'])
weekly_by_zone['week'] = weekly_by_zone['date'].dt.isocalendar().week
weekly_by_zone['year'] = weekly_by_zone['date'].dt.year

weekly_agg = weekly_by_zone.groupby(['year', 'week', 'zone'])['trips'].sum().reset_index()

# Plot top 6 zones
top_zones = df.groupby('zone')['trips'].sum().nlargest(6).index

fig, ax = plt.subplots(figsize=(14, 7))
for zone in top_zones:
    zone_data = weekly_agg[weekly_agg['zone'] == zone].reset_index(drop=True)
    ax.plot(range(len(zone_data)), zone_data['trips'], marker='o', label=zone, linewidth=2, markersize=4)

ax.set_title('Weekly Demand Trends by Zone (Top 6 Zones)', fontsize=16, fontweight='bold')
ax.set_xlabel('Week Number', fontsize=12)
ax.set_ylabel('Weekly Total Trips', fontsize=12)
ax.legend(loc='best', ncol=2)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'c02_weekly_trends_by_zone.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c02_weekly_trends_by_zone.png')

---
## C2 · Hourly Patterns

In [ ]:
print('=== C2: HOURLY PATTERN VISUALIZATIONS ===')

# Average demand by hour
hourly_avg = df.groupby('hour')['trips'].mean()

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(hourly_avg.index, hourly_avg.values, color=COLORS['primary'], alpha=0.8, edgecolor='black')

# Highlight peak hours
peak_hours = hourly_avg.nlargest(3).index
for i, bar in enumerate(bars):
    if i in peak_hours:
        bar.set_color(COLORS['accent'])
        bar.set_alpha(1.0)

ax.set_title('Average Hourly Demand Profile (Citywide)', fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Average Trips per Hour', fontsize=12)
ax.set_xticks(range(24))
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG / 'c03_hourly_demand_profile.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c03_hourly_demand_profile.png')

In [ ]:
# Hourly patterns by day of week (heatmap)
dow_names = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df.pivot_table(values='trips', index='hour', columns='dayofweek', aggfunc='mean')
pivot.columns = dow_names

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(pivot, cmap='YlOrRd', annot=True, fmt='.1f', cbar_kws={'label': 'Average Trips'}, ax=ax)
ax.set_title('Demand Heatmap: Hour × Day of Week', fontsize=16, fontweight='bold')
ax.set_xlabel('Day of Week', fontsize=12)
ax.set_ylabel('Hour of Day', fontsize=12)
plt.tight_layout()
plt.savefig(FIG / 'c04_hour_dow_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c04_hour_dow_heatmap.png')

In [ ]:
# Weekday vs Weekend patterns
hourly_weekday = df[df['is_weekend'] == 0].groupby('hour')['trips'].mean()
hourly_weekend = df[df['is_weekend'] == 1].groupby('hour')['trips'].mean()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(hourly_weekday.index, hourly_weekday.values, marker='o', linewidth=2.5, 
        label='Weekday', color=COLORS['primary'], markersize=6)
ax.plot(hourly_weekend.index, hourly_weekend.values, marker='s', linewidth=2.5, 
        label='Weekend', color=COLORS['secondary'], markersize=6)

ax.set_title('Hourly Demand: Weekday vs Weekend', fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Average Trips per Hour', fontsize=12)
ax.set_xticks(range(24))
ax.legend(loc='best', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'c05_weekday_vs_weekend.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c05_weekday_vs_weekend.png')

---
## C3 · Zone Comparisons

In [ ]:
print('=== C3: ZONE COMPARISON VISUALIZATIONS ===')

# Zone rankings (average demand)
zone_avg = df.groupby('zone')['trips'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 7))
colors = [COLORS['accent'] if i < 3 else COLORS['primary'] for i in range(len(zone_avg))]
bars = ax.barh(zone_avg.index, zone_avg.values, color=colors, edgecolor='black', alpha=0.85)

ax.set_title('Average Demand by Zone (Ranked)', fontsize=16, fontweight='bold')
ax.set_xlabel('Average Trips per Hour', fontsize=12)
ax.set_ylabel('Zone', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')

# Add value labels
for i, (idx, val) in enumerate(zone_avg.items()):
    ax.text(val + 0.5, i, f'{val:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig(FIG / 'c06_zone_rankings.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c06_zone_rankings.png')

In [ ]:
# Zone hourly profiles (spaghetti plot)
zone_hourly = df.groupby(['zone', 'hour'])['trips'].mean().reset_index()

fig, ax = plt.subplots(figsize=(14, 7))
for zone in sorted(df['zone'].unique()):
    zone_data = zone_hourly[zone_hourly['zone'] == zone]
    ax.plot(zone_data['hour'], zone_data['trips'], marker='o', linewidth=1.5, 
            label=zone, alpha=0.7, markersize=4)

ax.set_title('Hourly Demand Profiles by Zone', fontsize=16, fontweight='bold')
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Average Trips per Hour', fontsize=12)
ax.set_xticks(range(24))
ax.legend(loc='upper left', ncol=2, fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'c07_zone_hourly_profiles.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c07_zone_hourly_profiles.png')

In [ ]:
# Zone demand distribution (box plots)
top_zones = df.groupby('zone')['trips'].mean().nlargest(12).index
plot_data = df[df['zone'].isin(top_zones)].copy()

fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=plot_data, x='zone', y='trips', ax=ax, palette='Set2')
ax.set_title('Demand Distribution by Zone (Box Plots)', fontsize=16, fontweight='bold')
ax.set_xlabel('Zone', fontsize=12)
ax.set_ylabel('Trips per Hour', fontsize=12)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG / 'c08_zone_distribution_boxplots.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c08_zone_distribution_boxplots.png')

---
## C4 · Weather Impact

In [ ]:
print('=== C4: WEATHER IMPACT VISUALIZATIONS ===')

# Temperature vs demand (scatter + trend)
sample = df.sample(min(5000, len(df)), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Temperature
axes[0, 0].scatter(sample['temp_c'], sample['trips'], alpha=0.3, s=10, color=COLORS['primary'])
z = np.polyfit(sample['temp_c'].dropna(), sample['trips'][sample['temp_c'].notna()], 2)
p = np.poly1d(z)
x_smooth = np.linspace(sample['temp_c'].min(), sample['temp_c'].max(), 100)
axes[0, 0].plot(x_smooth, p(x_smooth), color=COLORS['accent'], linewidth=3, label='Trend')
axes[0, 0].set_xlabel('Temperature (°C)', fontsize=11)
axes[0, 0].set_ylabel('Trips', fontsize=11)
axes[0, 0].set_title('Temperature vs Demand', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Rain
rain_bins = [0, 0.1, 5, 10, 50]
rain_labels = ['No rain', 'Light (<5mm)', 'Moderate (5-10mm)', 'Heavy (>10mm)']
sample['rain_cat'] = pd.cut(sample['rain_mm'], bins=rain_bins, labels=rain_labels, include_lowest=True)
rain_avg = sample.groupby('rain_cat', observed=True)['trips'].mean()

axes[0, 1].bar(range(len(rain_avg)), rain_avg.values, color=COLORS['secondary'], alpha=0.8, edgecolor='black')
axes[0, 1].set_xticks(range(len(rain_avg)))
axes[0, 1].set_xticklabels(rain_avg.index, rotation=15, ha='right')
axes[0, 1].set_ylabel('Average Trips', fontsize=11)
axes[0, 1].set_title('Rainfall vs Demand', fontsize=12, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Humidity
axes[1, 0].scatter(sample['humidity_pct'], sample['trips'], alpha=0.3, s=10, color=COLORS['success'])
z = np.polyfit(sample['humidity_pct'].dropna(), sample['trips'][sample['humidity_pct'].notna()], 1)
p = np.poly1d(z)
x_smooth = np.linspace(sample['humidity_pct'].min(), sample['humidity_pct'].max(), 100)
axes[1, 0].plot(x_smooth, p(x_smooth), color=COLORS['warning'], linewidth=3, label='Trend')
axes[1, 0].set_xlabel('Humidity (%)', fontsize=11)
axes[1, 0].set_ylabel('Trips', fontsize=11)
axes[1, 0].set_title('Humidity vs Demand', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Wind
axes[1, 1].scatter(sample['wind_kmh'], sample['trips'], alpha=0.3, s=10, color=COLORS['neutral'])
z = np.polyfit(sample['wind_kmh'].dropna(), sample['trips'][sample['wind_kmh'].notna()], 1)
p = np.poly1d(z)
x_smooth = np.linspace(sample['wind_kmh'].min(), sample['wind_kmh'].max(), 100)
axes[1, 1].plot(x_smooth, p(x_smooth), color=COLORS['accent'], linewidth=3, label='Trend')
axes[1, 1].set_xlabel('Wind Speed (km/h)', fontsize=11)
axes[1, 1].set_ylabel('Trips', fontsize=11)
axes[1, 1].set_title('Wind Speed vs Demand', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Weather Effects on Ride Demand', fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(FIG / 'c09_weather_impact_grid.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c09_weather_impact_grid.png')

---
## C5 · Event Impact

In [ ]:
print('=== C5: EVENT IMPACT VISUALIZATIONS ===')

# Event vs non-event comparison
event_comparison = df.groupby('is_event')['trips'].agg(['mean', 'median', 'std']).reset_index()
event_comparison['is_event'] = event_comparison['is_event'].map({0: 'No Event', 1: 'Event'})

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart
x = np.arange(len(event_comparison))
width = 0.35

axes[0].bar(x - width/2, event_comparison['mean'], width, label='Mean', 
            color=COLORS['primary'], alpha=0.8, edgecolor='black')
axes[0].bar(x + width/2, event_comparison['median'], width, label='Median', 
            color=COLORS['secondary'], alpha=0.8, edgecolor='black')
axes[0].set_ylabel('Trips per Hour', fontsize=12)
axes[0].set_title('Event Impact: Mean vs Median', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(event_comparison['is_event'])
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Violin plot
event_data = df[df['is_event'] == 1]['trips'].dropna()
no_event_data = df[df['is_event'] == 0]['trips'].dropna()

parts = axes[1].violinplot([no_event_data, event_data], positions=[0, 1], 
                            showmeans=True, showmedians=True)
for pc in parts['bodies']:
    pc.set_facecolor(COLORS['primary'])
    pc.set_alpha(0.6)

axes[1].set_ylabel('Trips per Hour', fontsize=12)
axes[1].set_title('Event Impact: Distribution', fontsize=14, fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['No Event', 'Event'])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIG / 'c10_event_impact_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c10_event_impact_comparison.png')

In [ ]:
# Event type impact
event_cols = [c for c in df.columns if c.startswith('event_') and c not in ['is_event', 'n_events_active']]

event_type_impacts = []
for col in event_cols:
    event_type = col.replace('event_', '').replace('_', ' ').title()
    if df[col].sum() > 0:
        event_mean = df[df[col] == 1]['trips'].mean()
        baseline_mean = df[df['is_event'] == 0]['trips'].mean()
        lift_pct = (event_mean - baseline_mean) / baseline_mean * 100
        
        event_type_impacts.append({
            'event_type': event_type,
            'mean_trips': event_mean,
            'lift_%': lift_pct
        })

event_impacts_df = pd.DataFrame(event_type_impacts).sort_values('lift_%', ascending=False)

if len(event_impacts_df) > 0:
    fig, ax = plt.subplots(figsize=(12, 7))
    colors_list = [COLORS['success'] if x > 0 else COLORS['warning'] for x in event_impacts_df['lift_%']]
    bars = ax.barh(event_impacts_df['event_type'], event_impacts_df['lift_%'], 
                   color=colors_list, alpha=0.8, edgecolor='black')
    
    ax.axvline(x=0, color='black', linewidth=1.5, linestyle='--')
    ax.set_xlabel('Demand Lift (%)', fontsize=12)
    ax.set_title('Event Type Impact on Demand', fontsize=16, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='x')
    
    # Add value labels
    for i, (idx, row) in enumerate(event_impacts_df.iterrows()):
        val = row['lift_%']
        ax.text(val + 1, i, f"{val:+.1f}%", va='center', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(FIG / 'c11_event_type_impact.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print('✓ Saved: c11_event_type_impact.png')
else:
    print('⚠ No event data to visualize')

---
## C6 · Correlation Heatmap

In [ ]:
print('=== C6: CORRELATION HEATMAP ===')

# Select key features for correlation
corr_features = [
    'trips', 'hour', 'dayofweek', 'is_weekend', 'month',
    'temp_c', 'rain_mm', 'humidity_pct', 'wind_kmh',
    'is_event', 'active_drivers'
]

# Filter to available columns
available_features = [f for f in corr_features if f in df.columns]
corr_matrix = df[available_features].corr()

# Create figure
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={'label': 'Correlation'},
            ax=ax, vmin=-1, vmax=1)

ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG / 'c12_correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c12_correlation_heatmap.png')

---
## C7 · Distribution Plots

In [ ]:
print('=== C7: DISTRIBUTION PLOTS ===')

# Trip count distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Histogram
axes[0].hist(df['trips'].dropna(), bins=50, color=COLORS['primary'], alpha=0.7, edgecolor='black')
axes[0].axvline(df['trips'].mean(), color=COLORS['accent'], linewidth=2, linestyle='--', label=f'Mean: {df["trips"].mean():.1f}')
axes[0].axvline(df['trips'].median(), color=COLORS['warning'], linewidth=2, linestyle='--', label=f'Median: {df["trips"].median():.1f}')
axes[0].set_xlabel('Trips per Hour', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Hourly Trip Counts', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Log-scale histogram
axes[1].hist(df['trips'].dropna(), bins=50, color=COLORS['secondary'], alpha=0.7, edgecolor='black')
axes[1].set_xlabel('Trips per Hour', fontsize=12)
axes[1].set_ylabel('Frequency (log scale)', fontsize=12)
axes[1].set_title('Distribution (Log Scale)', fontsize=14, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.savefig(FIG / 'c13_trip_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c13_trip_distribution.png')

In [ ]:
# Q-Q plot (check normality)
from scipy import stats

fig, ax = plt.subplots(figsize=(8, 8))
stats.probplot(df['trips'].dropna(), dist="norm", plot=ax)
ax.set_title('Q-Q Plot: Trip Counts vs Normal Distribution', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / 'c14_qq_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Saved: c14_qq_plot.png')

---
## Summary

**Visualizations complete!** Generated figures:

1. **Time Series** (C1)
   - c01_daily_demand_timeseries.png
   - c02_weekly_trends_by_zone.png

2. **Hourly Patterns** (C2)
   - c03_hourly_demand_profile.png
   - c04_hour_dow_heatmap.png
   - c05_weekday_vs_weekend.png

3. **Zone Comparisons** (C3)
   - c06_zone_rankings.png
   - c07_zone_hourly_profiles.png
   - c08_zone_distribution_boxplots.png

4. **Weather Impact** (C4)
   - c09_weather_impact_grid.png

5. **Event Impact** (C5)
   - c10_event_impact_comparison.png
   - c11_event_type_impact.png

6. **Correlations** (C6)
   - c12_correlation_heatmap.png

7. **Distributions** (C7)
   - c13_trip_distribution.png
   - c14_qq_plot.png

All figures saved to `figures/` directory at 300 DPI for publication quality.

**Next:** Build predictive models in `04_modeling_and_evaluation.ipynb`